In [1]:
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import requests
import json
from bs4 import BeautifulSoup, Tag
from bs4.element import NavigableString
import re

import undetected_chromedriver as uc
import time

from collections import deque, Counter
from pymongo import MongoClient

from datetime import datetime, timezone

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer     # for stemming
from nltk.tokenize import word_tokenize
import string

import math



Web Crawler data flow: 

1. Take seed URL from frontier and request IP from DNS
2. Fetch HTML from external server using IP
3. Extract text data from the HTML.
4. Store the text data in a database.
5. Extract any linked URLs from the web pages and add them to the list (Frontier Queue) of URLs to crawl.
6. Repeat steps 1-5 until all URLs have been crawled.

In [2]:
# TESTING_SEED_URL = "https://softwarica.edu.np/courses"
# TESTING_ALLOWED_DOMAIN = "softwarica.edu.np"

# TESTING_CRAWL_DELAY_SECONDS = 5*24*60*60      # 5 days
# TESTING_MAX_PAGES = 100               
# TESTING_USER_AGENT = "SoftwaricaVerticalSearchBot/1.0 (+educational IR project)"

In [3]:
SEED_URL = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/"

In [4]:
# SEED_URL = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/"
ALLOWED_DOMAIN = "pureportal.coventry.ac.uk"
CRAWL_DELAY_SECONDS = 24*60*60*30*3    # 3 months      
MAX_PAGES = 100               
USER_AGENT = "CoventryVerticalSearchBot/1.0 (+educational IR project)"
my_current_chrome_version = 146


Ensuring Politness steps: 

1. To make this clear, the steps would be:
2. Fetch the `robots.txt` file for the domain.
3. Parse the `robots.txt` file and store it in the database (MongoDB).
4. When we pull a URL off the queue, check the rules stored in the database (MongoDB) for that domain.
5. If the URL is disallowed, ack the message and move on to the next URL.
6. If the URL is allowed, check the `Crawl-delay` directive.

IF Crawler is failed

7. If the Crawl-delay time has not passed since the last crawl, use ChangeMessageVisibility to extend the visibility timeout and defer reprocessing.
8. If the Crawl-delay time has passed, crawl the page and update the last crawl time for the domain.

In [5]:
db_name = "vertical_search_engine_cw"

local_mongo_url = "mongodb://localhost:27017"
client = MongoClient(local_mongo_url)
db = client[db_name]

# for raw pages of publications (research output)
raw_pages_publications = db["raw_pages_publications"]

# for raw pages of profiles 
raw_pages_profiles = db["raw_pages_profiles"]

# for vectors of each document
doc_vectors = db["doc_vectors"]
# for IDF
term_index = db["term_index"]
# for storing crawled logs
crawl_log = db["crawl_log"]

# for storing raw pages of publications and profiles 
raw_pages_publications.create_index("url", unique=True)
raw_pages_profiles.create_index("url", unique=True)

# for storing normalized TF-IDF vector per document
doc_vectors.create_index("url", unique=True)

# for storing IDF value per word (title, authors, journal name, volume, number of pages, publish date)
term_index.create_index("term", unique=True)

print("Connected:", db.name, "| collections:", db.list_collection_names())


Connected: vertical_search_engine_cw | collections: ['doc_vectors', 'raw_pages_publications', 'term_index', 'raw_pages_profiles']


In [6]:
# fetching the content of the robots.txt file 

def fetch_robots(base_url, USER_AGENT):
    parsed_url = urlparse(base_url)     # Example: ParseResult(scheme='https', netloc='softwarica.edu.np', path='/courses', params='', query='', fragment='')
    robots_url = f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"
    rfp = RobotFileParser()
    rfp.set_url(robots_url)

    try: 
        res = requests.get(robots_url, headers={"User-Agent": USER_AGENT}, timeout=10)
        if res.status_code == 200: 
            rfp.parse(res.text.splitlines())
        else: 
            rfp = None
    except BaseException as err: 
        print(f"Error: {err}")
        rfp = None
        
    return rfp

In [7]:
def can_fetch(rfp, url, USER_AGENT=USER_AGENT):
    if rfp is None: 
        return True
    return rfp.can_fetch(USER_AGENT, url)

In [8]:

# 
def setup_driver(current_chrome_version):
    # getting chrome options 
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options, version_main=current_chrome_version)
    return driver
    


In [9]:
# bypassing the Cloudflare bot by using undetected-chromedriver
driver = setup_driver(my_current_chrome_version)

driver.get(SEED_URL)

time.sleep(10)

html_content = driver.page_source
soup = BeautifulSoup(html_content, "html.parser")

print("Page loaded. Extracting...\n")

Page loaded. Extracting...



Note: Rate limiting (avoiding system crash by requesting to crawl) is important. Sliding window algorithm can be used to track the number of requests per domain per second

In [10]:
def extract_research_output(base_url, driver):
    
    print(f"\n{"="*20} Publications HTML extraction initialized {"="*20}\n")
    

    # for all research output data
    extracted_data = []

    # tracking page number since, research ouptut is divided into page 1, 2
    page_number = 0

    while True:
        # making paginated url
        current_url = f"{base_url}publications/?page={page_number}"
        print(f"Fetching: {current_url}")

        driver.get(current_url)

        # waiting for cloudflare on the first page, subsequent pages might load faster
        if page_number == 0:
            time.sleep(10)
        else:
            # waiting shorter for subsequent pages assuming cloudflare is already passed
            time.sleep(5)

        # extracting all HTML content of the page
        html_content = driver.page_source
        soup = BeautifulSoup(html_content, "html.parser")

        # finding all research outputs on the current page
        results = soup.find_all('div', class_='rendering_researchoutput')

        # if no results are found on this page, we have reached the end
        if not results:
            print(f"No more results found. Exiting loop.")
            break

        print(f"Found {len(results)} outputs on page {page_number}. Extracting...\n")

        for div in results:
            title = None
            title_link = None

            # extracting title and its link
            h3 = div.find('h3', class_='title')
            if h3:
                a_tag = h3.find('a', class_='link')
                if a_tag:
                    title = a_tag.text.strip()
                    title_link = a_tag.get('href')
                else:
                    # fallback in case there is no link
                    title = h3.text.strip()

            # extracting authors along with link
            authors = []
            if h3:
                # authors are floating between the h3 tag and the date span
                for sibling in h3.next_siblings:
                    # stop looking for authors once we hit the date span
                    if isinstance(sibling, Tag) and sibling.name == 'span':
                        sibling_class = sibling.get('class')
                        if sibling_class and 'date' in sibling_class:
                            break

                    # if the sibling is raw text
                    if isinstance(sibling, NavigableString) and not isinstance(sibling, Tag):
                        # cleaning up the raw text to remove dangling commas and whitespace
                        text = sibling.strip(', & \n\r\t')
                        if text:
                            authors.append({'name': text, 'link': None})

                    # if the sibling is an a tag
                    elif isinstance(sibling, Tag) and sibling.name == 'a':
                        sibling_class = sibling.get('class')
                        if sibling_class and 'person' in sibling_class:
                            authors.append({
                                'name': sibling.text.strip(),
                                'link': sibling.get('href')
                            })

            # extracting publish date
            date_span = div.find('span', class_='date')
            publish_date = date_span.text.strip() if date_span else None

            # extracting journal name
            journal_span = div.find('span', class_='journal')
            journal_name = journal_span.text.strip() if journal_span else None

            # extracting journal volume
            volume_span = div.find('span', class_='volume')
            journal_volume = volume_span.text.strip() if volume_span else None

            # extracting number of pages
            pages_span = div.find('span', class_='numberofpages')
            number_of_pages = pages_span.text.strip() if pages_span else None

            research_output = {
                'title': title,
                'title_link': title_link,
                'authors': authors,
                'publish_date': publish_date,
                'journal_name': journal_name,
                'journal_volume': journal_volume,
                'number_of_pages': number_of_pages
            }

            if research_output['title'] is None:
                continue

            # store each research output as its own document, keyed by title/link
            document_to_store = {
                **research_output,
                'url': title_link if title_link else f"{current_url}#{title}", 'source_page_url': current_url,
                'crawled_at': datetime.utcnow()
            }

            key = {"title": title}
            if title_link:
                key = {"title_link": title_link}

            raw_pages_publications.update_one(
                key,
                {"$set": document_to_store},
                upsert=True
            )
            extracted_data.append(research_output)

        # adding page number to go to the next page
        page_number += 1

    # driver.quit()
    # print("Driver is closed...")

    # saving json file
    output_filename = "all_research_outputs.json"
    # with open(f"../data/{output_filename}", 'w', encoding='utf-8') as json_file:
    #     json.dump(extracted_data, json_file, indent=4, ensure_ascii=False)

    print(f"\nSuccessfully saved a total of {len(extracted_data)} research outputs to '{output_filename}'!")
    
    print(f"\n{"="*20} Publications HTML extraction closed {"="*20}\n")
    

    return extracted_data

In [11]:
# extract_research_output(SEED_URL)

In [12]:
def extract_profiles(base_url, driver):
    
    print(f"\n{"="*20} Profiles HTML extraction initialized {"="*20}\n")
    

    # storing all data across all pages here
    extracted_profiles = []

    # starting pagination at page 0
    page_number = 0

    while True:
        # constructing the paginated url
        current_url = f"{base_url}persons/?page={page_number}"
        print(f"Fetching: {current_url}")
        
        driver.get(current_url)
        
        
        # waiting for cloudflare on the first page, subsequent pages might load faster
        if page_number == 0:
            time.sleep(10)
        else:
            # waiting a shorter time for subsequent pages assuming cloudflare is already passed
            time.sleep(5)
            
        html_content = driver.page_source
        soup = BeautifulSoup(html_content, "html.parser")
        
        # targeting the main container for each profile card
        results = soup.find_all('div', class_='result-container')
        
        # breaking the loop if no results are found on this page
        if not results:
            print("No more profiles found. Exiting loop.")
            break
            
        print(f"Found {len(results)} profiles on page {page_number}. Extracting...\n")
        
        if page_number == 3:
            break
        
        for div in results:
            # skipping if this container doesn't actually hold a person profile
            if not div.find('div', class_='rendering_person'):
                continue
                
            # extracting image url
            image_url = None
            img_tag = div.find('img', class_='image')
            if img_tag and img_tag.get('src'):
                image_url = img_tag.get('src')
                # appending the base domain because pureportal often uses relative image paths
                if image_url and isinstance(image_url, str) and image_url.startswith('/'):
                    image_url = f"https://pureportal.coventry.ac.uk{image_url}"

            # extracting name and profile link
            name = None
            profile_link = None
            h3 = div.find('h3', class_='title')
            if h3:
                a_tag = h3.find('a')
                if a_tag:
                    name = a_tag.text.strip()
                    profile_link = a_tag.get('href')
                else:
                    name = h3.text.strip()

            # extracting relations / organisations
            organizations = []
            org_ul = div.find('ul', class_='relations organisations')
            if org_ul:
                # finding all list items within the relations ul
                for li in org_ul.find_all('li'):
                    org_text = li.text.strip()
                    if org_text:
                        organizations.append(org_text)

            # extracting person type (e.g., academic staff)
            type_p = div.find('p', class_='type')
            person_type = type_p.text.strip() if type_p else None

            # extracting active publication years (from the stacked-trend-widget)
            start_year = None
            end_year = None
            years = div.find_all('span', class_='stacked-trend-graph-year')
            
            if years:
                # assigning the first span as the start year
                start_year = years[0].text.strip()
                # assigning the last span as the end year (if there is more than one year)
                if len(years) > 1:
                    end_year = years[-1].text.strip()
                else:
                    # setting the end year same as start year if only one year is listed
                    end_year = start_year 

            # compiling into a dictionary
            profile_data = {
                'name': name,
                'profile_link': profile_link,
                'image_url': image_url,
                'organizations': organizations,
                'person_type': person_type,
                'active_years': {
                    'start': start_year,
                    'end': end_year
                }
            }

            # preparingto store profile data in DB
            document_to_store = {
                **profile_data,
                'url': profile_link if profile_link else f"{current_url}#{name}", 'source_page_url': current_url,
                'crawled_at': datetime.utcnow()
            }

            profile_key = {"name": name}
            if profile_link:
                profile_key = {"profile_link": profile_link}

            # storing profiles data
            raw_pages_profiles.update_one(
                profile_key,
                {"$set": document_to_store},
                upsert=True
            )


            extracted_profiles.append(profile_data)



        # incrementing page number to go to the next page    #     

        page_number += 1    

    output_filename = "all_profiles.json"
    
    # with open(f"../data/{output_filename}", 'w', encoding='utf-8') as json_file:
        #     json.dump(extracted_data, json_file, indent=4, ensure_ascii=False)

    print(f"\nSuccessfully saved a total of {len(extracted_profiles)} profiles to '{output_filename}'!")

    # closing the browser

    # driver.quit()
    
    print(f"\n{"="*20} Profiles HTML extraction closed {"="*20}\n")
    
    
    return extracted_profiles

In [13]:

# # defining a crawler

# def crawl(seed_url, max_pages, user_agent, allowed_domain, crawl_delay_seconds, current_chrome_extension):
#     """crawls through the entire links of the seed url and returns the number of crawled links

#     Args:
#         seed_url (str): seed url 
#         max_pages (int): up to max pages for crawling
#         user_agent (str): name of user agent
#         allowed_domain (str): allowed domain to crawl
#         crawl_delay_seconds (time): time (seconds) in integer for ensuring politness

#     Returns:
#         int: number of crawled links
#     """
    
#     rfp = fetch_robots(seed_url, user_agent)
    
#     # storing all extracted URLs to visit
#     frontier_queue = deque([seed_url])
    
#     visited_links = set()  # not visited any link so empty
    
#     crawled_count = 0  # not crawled yet so empty
    
#     print(f"{10*"="}Initializing browser and bypassing Cloudflare {10*"="}")
#     driver = setup_driver(my_current_chrome_version)

#     try: 
#         # loop until all links are visited
#         while frontier_queue and crawled_count < max_pages:
            
#             url = frontier_queue.popleft()
#             print('URL', url)
            
#             if url in visited_links:
#                 continue
            
#             if not can_fetch(rfp, url):
#                 print(f"Blocked by robots.txt: {url}")
#                 continue
            
#             try:
#                 driver.get(url)
#                 # wait for 5 sec between requests (ensuring politness --> DO NOT hit the host server)
                
#                 time.sleep(10)
#                 # WebDriverWait(driver, 10).until(
#                 #     ec.presence_of_element_located((By.TAG_NAME, "body"))
#                 # )
                
#                 html_content = driver.page_source
                
#             except BaseException as err:
#                     print(f"Error while fetching {url} url: {err}")
#                     continue
#             research_outputs = extract_research_output(seed_url)
            
#             profiles = extract_profiles(seed_url)
            
#             # TODO: store raw pages in DB
            
            
            
#             crawled_count += 1
            
#             print(f"{crawled_count} Crawled: {url}")
            
#             for link in links:
#                 # if link is not visited then add it to the url frontier queue
#                 if link not in visited_links:
#                     frontier_queue.append(link)
            
#                 # wait for 5 sec between request (ensuring politness --> DO NOT hit the host server)
            
#             time.sleep(crawl_delay_seconds)
            
#     except BaseException as err:
#         print(f"Error while crawling: {err}")
        
#     finally:
#         driver.quit()
#         print(f"{10*"="} Closed Headless Chrome driver {10*"="}")
        
#     # TODO: store crawl logs in DB
#     print(f"Crawling completed. {crawled_count} pages crawled and needs to be stored in DB")
#     return crawled_count
    

In [14]:
def get_links(html_content, base_url):
    """
    fetching all links from the base url and filtering out only required links for research output publications and persons (profile)
    """
    target_links = {}
    soup = BeautifulSoup(html_content, "html.parser")
    
    for a in soup.find_all("a", href=True):
        # cleaning the link by removing fragments and query parameters
        href = a["href"]
        if isinstance(href, str):
            clean_link = urljoin(base_url, href).split("#")[0].split("?")[0]
            
            if clean_link.startswith(base_url):
                # using rstrip("/") to safely check the end of the URL regardless of a trailing slash
                if clean_link.rstrip("/").endswith("publications"):
                    target_links["publications"] = clean_link
                    
                elif clean_link.rstrip("/").endswith("persons"):
                    target_links["persons"] = clean_link
                
    for category, link in target_links.items():
        print(f"Found {category}: {link}")
        
    return target_links

def crawl(seed_url, user_agent, current_chrome_version):
    """ crawls through the entire links of the seed url, saves raw data of publications and profiles in DB and then, returns the number of crawled links

    Args:
          seed_url (str): seed url 
          user_agent (str): name of user agent
          current_chrome_version (int): takes current version of chrome version


    Returns:
        int: number of crawled links
    """
    print(f"\n{"="*20} Crawling initialization {"="*20}\n")
    
    visited_links = set()
    # checking robots.txt permissions first
    rfp = fetch_robots(seed_url, user_agent)
    if not can_fetch(rfp, seed_url):
        print(f"Blocked by robots.txt: {seed_url}")
        return 0
        
    print(f"{10*'='} Initializing browser and bypassing Cloudflare {10*'='}")
    driver = setup_driver(current_chrome_version)
    
    try:
        # navigating to the main seed url (the organisation home page)
        print(f"Fetching seed URL: {seed_url}")
        driver.get(seed_url)
        
        # waiting for Cloudflare and dynamic JS to load
        time.sleep(10)
        html_content = driver.page_source
        
        
        publications_url = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/"
        profiles_url = "https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/persons/"
        
        crawled_count = 0
        
        # triggering the research output extraction if the link was found
        if publications_url:
            print(f"\n--- Starting Publications Extraction ---")
            # passing the specific publications URL to your extractor
            research_outputs = extract_research_output(seed_url, driver)
            crawled_count += len(research_outputs)
            visited_links.add(publications_url)
                
            
        # triggering the profiles extraction if the link was found
        if profiles_url:
            print(f"\n--- Starting Profiles Extraction ---")
            # passing the specific profiles URL to your extractor
            profiles = extract_profiles(seed_url, driver)
            crawled_count += len(profiles)
            visited_links.add(profiles_url)

    except BaseException as err:
        print(f"Error while crawling: {err}")
        
    finally:
        # closing the driver safely
        driver.quit()
        print(f"{10*'='} Closed Headless Chrome driver {10*'='}")
        
    print(f"Crawling completed. {crawled_count} total items extracted and saved to JSON.")
    
    print(f"\n{"="*20} Crawling closed {"="*20}\n")
    
    return crawled_count

In [15]:

# crawl coventry pure potal

crawl(
    seed_url=SEED_URL,
    user_agent=USER_AGENT,
    current_chrome_version=my_current_chrome_version
)


==================== Crawling initialization ====================

========== Initializing browser and bypassing Cloudflare ==========
Fetching seed URL: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/

--- Starting Publications Extraction ---

==================== Publications HTML extraction initialized ====================

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/?page=0
Found 96 outputs on page 0. Extracting...

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/?page=1


C:\Users\shres\AppData\Local\Temp\ipykernel_13484\1749824163.py:115: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'crawled_at': datetime.utcnow()


Found 53 outputs on page 1. Extracting...

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/publications/?page=2
No more results found. Exiting loop.

Successfully saved a total of 79 research outputs to 'all_research_outputs.json'!

==================== Publications HTML extraction closed ====================


--- Starting Profiles Extraction ---

==================== Profiles HTML extraction initialized ====================

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/persons/?page=0
Found 50 profiles on page 0. Extracting...

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/persons/?page=1
Found 50 profiles on page 1. Extracting...

Fetching: https://pureportal.coventry.ac.uk/en/organisations/centre-for-healthcare-and-community-transformation/persons/?page=2
Found 17 profiles on page 2. Extr

196

In [16]:
# preprocessing the text by applying lowercase, strip non-alphanumerics, tokenize, drop stopwords, and stemming
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = word_tokenize(text)
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens if word not in stopwords.words("english") and word not in string.punctuation and len(word) > 2]
    return " ".join(tokens)

In [19]:
# building the TF-IDF weighted document vectors and the inverted index for terms (title, authors, journal name, volume, number of pages, publish date)

def do_indexing_and_saving():
    """Calculates IDF, and TF-IDF and save them in 'term_index' and 'doc_vectors' respectively
    """
    
    print(f"\n{"="*20} Indexing initialization {"="*20}\n")
    
    docs = list(raw_pages_publications.find({}))
    # total number of publications document
    total_docs =  len(docs)
    if total_docs == 0:
        print("No documents found to index.")
        return
        
    doc_tokens = {}
    df = Counter()
    for doc in docs:
        title_tokens = preprocess(doc.get("title"))
        authors_tokens = " ".join([preprocess(author.get("name", "")) for author in doc["authors"]])
        
        # journal name, volum, and number of pages can be null so if they are empty then an empty [] otherwise preprocess them
        journal_name_tokens = preprocess(doc.get("journal_name")) if doc.get("journal_name") else []
        journal_volume_tokens = preprocess(doc.get("journal_volume")) if doc.get("journal_volume") else []
        number_of_pages_tokens = preprocess(doc.get("number_of_pages")) if doc.get("number_of_pages") else []
        publish_date_tokens = preprocess(doc.get("publish_date")) if doc.get("publish_date") else []
        
        
        all_doc_string = " ".join([
            title_tokens if isinstance(title_tokens, str) else " ".join(title_tokens),
            authors_tokens,
            journal_name_tokens if isinstance(journal_name_tokens, str) else " ".join(journal_name_tokens),
            journal_volume_tokens if isinstance(journal_volume_tokens, str) else " ".join(journal_volume_tokens),
            number_of_pages_tokens if isinstance(number_of_pages_tokens, str) else " ".join(number_of_pages_tokens),
            publish_date_tokens if isinstance(publish_date_tokens, str) else " ".join(publish_date_tokens)
        ])
        
        word_list = all_doc_string.split()
        
        # extracting URL
        doc_url = doc.get("url")
        if doc_url: 
            # storing the actual list of words for the document
            doc_tokens[doc_url] = word_list
        
        # updating the Document Frequency (DF) counter and applying set() to ensure a word is only counted once per document for the IDF formula
        for term in set(word_list):
            df[term] += 1

    # calculating IDF for each word using the populated df counter
    idf = {term: math.log10(total_docs / (1 + freq)) + 1 for term, freq in df.items()}

    print(f"Processed {total_docs} documents. Vocabulary size: {len(idf)} unique terms.")


    # storing IDF in DB

    if idf:

        term_index.delete_many({})  # before inserting, deleting all older IDFs
        print("Deleted older IDF values from 'term_index' collection")
        
        # inserting the new IDFs
        term_index.insert_many([
            {
                "term": word, 
                "idf": idf_value,
                
            } for word, idf_value in idf.items()
        ]) 
        
        print("Inserted IDF values in 'term_index' collection")
        

    # storing TF-IDF in DB
    doc_vectors.delete_many({})  # deleting older values from DB
    print("Deleted older vecotrs document from 'doc_vectors' collection")

    #  calculating TF-IDF vectors and normalizing them for cosine similarity
    for doc in docs:
        
        url = doc.get("url")
        if not url:
            continue
        
        # fetching the clean list of words (tokens) which is previously processed 
        tokens = doc_tokens.get(url, []) 
        
        # counting total number of words otherwise, 1 if they are counted as 0
        total_terms = len(tokens) or 1
        
        # creating frequency dict to count how many times each unique word appeared
        tf = Counter(tokens)    
        
        # calculating the raw TF-IDF weight for each word
        vector = {term: (count / total_terms) * idf.get(term, 0) for term, count in tf.items()}
        
        # calculating the L2 norm (Euclidean length) of the document's vector
        norm = math.sqrt(sum(w * w for w in vector.values())) or 1.0
        
        # normalizing the document vector (TF-IDF)
        vector = {t: w / norm for t, w in vector.items()}
        
        # preparing the document payload for MongoDB
        vector_document = {
            "url": url,
            # keeping the title alongside the vector makes generating search results much faster later
            "title": doc.get("title", ""), 
            "vector": vector,
            # generating a timezone-aware UTC datetime for the timestamp
            "indexed_at": datetime.utcnow()
        }
        
        
        # upserting the normalized vector into the doc_vectors collection
        doc_vectors.update_one(
            {"url": url},
            {"$set": vector_document},
            upsert=True
        )
        
    print(f"Successfully calculated and stored {len(docs)} document vectors in the database.")
    print(f"\n{"="*20} Indexing closed {"="*20}\n")
    
        



In [20]:
do_indexing_and_saving()


==================== Indexing initialization ====================

Processed 79 documents. Vocabulary size: 970 unique terms.
Deleted older IDF values from 'term_index' collection
Inserted IDF values in 'term_index' collection
Deleted older vecotrs document from 'doc_vectors' collection
Successfully calculated and stored 79 document vectors in the database.

==================== Indexing closed ====================



C:\Users\shres\AppData\Local\Temp\ipykernel_13484\1151842770.py:111: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "indexed_at": datetime.utcnow()


In [18]:
# # Re-runs the crawl and re-indexing once a week
# def scheduled_job():
#     print(f"Running scheduled weekly crawl at {datetime.now()}")
#     crawl()
#     build_index()   

# schedule.every(7).days.do(scheduled_job)

# def run_scheduler_blocking():
#     """Run this in a separate long-lived process, not inside the notebook kernel."""
#     while True:
#         schedule.run_pending()
#         time.sleep(60)